# 02 — Verify All Secrets and Storage Connection
**Day 2 | Part 3 + Part 7**

Confirms all Key Vault secrets are readable from Databricks and tests the zero-credential storage pattern.

**Prerequisites:**
- Key Vault secret scope `kv-ev-scope` exists (Day 1 Part 6.5)
- All secrets added to Key Vault (Day 2 Part 3)
- Event Hub secrets added (Day 2 Part 4)

## Cell 1 — List all secrets in scope

In [ ]:
SCOPE = "kv-ev-scope"

print("Available secret scopes:")
for s in dbutils.secrets.listScopes():
    print(f"  {s.name}")

print(f"\nSecrets in {SCOPE}:")
try:
    for s in dbutils.secrets.list(SCOPE):
        print(f"  {s.key}")
except Exception as e:
    print(f"  ERROR listing secrets: {e}")
    print("  Check that kv-ev-scope is created and linked to Key Vault")

## Cell 2 — Verify all required secrets are readable

In [ ]:
SCOPE = "kv-ev-scope"

required_secrets = [
    "adls-account-name",
    "sp-client-id",
    "sp-client-secret",
    "sp-tenant-id",
    "voltgrid-api-base-url",
    "voltgrid-username",
    "voltgrid-password",
    "eventhub-connection-string",
    "eventhub-namespace",
    "eventhub-name",
]

print(f"{'Secret':<35} {'Status':>10}")
print("-" * 48)
missing = []

for key in required_secrets:
    try:
        dbutils.secrets.get(scope=SCOPE, key=key)
        print(f"  {key:<33} OK")
    except Exception:
        print(f"  {key:<33} MISSING")
        missing.append(key)

print("-" * 48)
if missing:
    print(f"\nMissing secrets ({len(missing)}): {missing}")
    print("Add them: Portal → kv-ev-intelligence-dev → Secrets → + Generate/Import")
else:
    print(f"\nAll {len(required_secrets)} secrets verified — OK")

## Cell 3 — Configure storage auth (SP OAuth — no mount)

In [ ]:
SCOPE = "kv-ev-scope"

storage_account  = dbutils.secrets.get(scope=SCOPE, key="adls-account-name")
sp_client_id     = dbutils.secrets.get(scope=SCOPE, key="sp-client-id")
sp_client_secret = dbutils.secrets.get(scope=SCOPE, key="sp-client-secret")
sp_tenant_id     = dbutils.secrets.get(scope=SCOPE, key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", sp_client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", sp_client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

def abfss(container: str, path: str = "") -> str:
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else base

print(f"Storage account : {storage_account}")
print("Storage OAuth auth configured — OK")

## Cell 4 — Storage write / read / delete test

In [ ]:
test_path = abfss("bronze", "api/_day2_test.txt")

try:
    dbutils.fs.put(test_path, "day2 secrets verification ok", overwrite=True)
    content = dbutils.fs.head(test_path)
    dbutils.fs.rm(test_path)
    print(f"Write  : OK")
    print(f"Read   : OK — '{content}'")
    print(f"Delete : OK")
    print("Storage write/read/delete — PASSED")
except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  403 Forbidden              → SP missing Storage Blob Data Contributor on storage")
    print("  AuthenticationFailed       → SP client secret wrong or expired in Key Vault")
    raise

## Cell 5 — Event Hub secrets check

In [ ]:
try:
    conn_str  = dbutils.secrets.get(scope=SCOPE, key="eventhub-connection-string")
    namespace = dbutils.secrets.get(scope=SCOPE, key="eventhub-namespace")
    eh_name   = dbutils.secrets.get(scope=SCOPE, key="eventhub-name")

    print(f"Event Hub namespace : {namespace}")
    print(f"Event Hub topic     : {eh_name}")
    print(f"Connection string   : {conn_str[:40]}...[REDACTED]")
    print("Event Hub secrets — OK")
except Exception as e:
    print(f"ERROR: {e}")
    print("Add Event Hub secrets to Key Vault — Part 4 of Day 2 setup guide")